In [0]:
spark.conf.set(
    "fs.azure.account.key.azsynapsestudy.dfs.core.windows.net",
    ""
)

In [0]:
EQ_silver_raw_df = spark.read.format("Delta").option("inferschema", "True").load("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/bronze")


In [0]:
from pyspark.sql.functions import col

EQ_silver_final_df = EQ_silver_raw_df.withColumn("date_time", (col("time") / 1000).cast("timestamp"))
EQ_silver_raw_df = EQ_silver_final_df.withColumn("Updated_date_time", (col("updated") / 1000).cast("timestamp"))
EQ_silver_final_df.show(1)

+-------+---+--------------------+-------------+-------------+----+--------------------+--------------------+----+----+----+-----+--------+-------+---+---+--------+-------+--------------------+----+-----+----+---+-------+---------------+--------------------+-------------+--------+---------+--------+----------+--------------------+-------------------+
|   type|mag|               place|         time|      updated|  tz|                 url|              detail|felt| cdi| mmi|alert|  status|tsunami|sig|net|    code|sources|               types| nst| dmin| rms|gap|magType|properties_type|               title|geometry_type|latitude|longitude|   depth|        id|           date_time|  Updated_date_time|
+-------+---+--------------------+-------------+-------------+----+--------------------+--------------------+----+----+----+-----+--------+-------+---+---+--------+-------+--------------------+----+-----+----+---+-------+---------------+--------------------+-------------+--------+---------+---

In [0]:
EQ_silver_final_df.drop(col("time")).drop(col("updated"))


DataFrame[type: string, mag: string, place: string, tz: string, url: string, detail: string, felt: string, cdi: string, mmi: string, alert: string, status: string, tsunami: int, sig: string, net: string, code: string, sources: string, types: string, nst: string, dmin: string, rms: string, gap: string, magType: string, properties_type: string, title: string, geometry_type: string, latitude: string, longitude: string, depth: string, id: string, date_time: timestamp, Updated_date_time: timestamp]

In [0]:
EQ_silver_final_df.select(col("sources")).distinct().show()

+----------+
|   sources|
+----------+
|        ld|
|        pr|
|        us|
|        pt|
|ismpkansas|
|        nm|
|        ci|
|        uw|
|        nn|
|     atlas|
|    iscgem|
|        nc|
|        at|
|        ak|
|        mb|
|          |
|        se|
|        uu|
|        hv|
+----------+



In [0]:
from pyspark.sql.functions import initcap,upper,split, explode,trim

EQ_silver_final_df = EQ_silver_final_df.withColumn("status", initcap("status"))\
    .withColumn("properties_type", initcap("properties_type"))\
    .withColumn("net", upper("net"))\
    .withColumn("type", initcap("type"))\
    .withColumn("magType", initcap("magType"))\
    .withColumn("alert", initcap("alert"))\
    .withColumn("sources", explode(split(trim("sources"), ",")))\
    .withColumn("sources", trim(col("sources"))) \
    .filter(col("sources") != "")\
    .dropDuplicates()




In [0]:
EQ_silver_final_df.write.format("delta").mode("overwrite").save("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/silver")

In [0]:
EQ_silver_final_df.count()

18681